# TT-Coach - Phase 2 Pipeline

**Video in -> analysed shots, rallies, and a rendered video out.**

Self-contained: every fix from Phase 2 development is folded in. This supersedes notebooks 09, 10 and 11.

```
video.mp4
  1  activity gate      both players present and moving  (sequential decode)
  2  pose               RTMPose-l on detector crops, active regions only
  3  canonicalise       hip-centred, torso-scaled, side-mirrored
  4  contact detection  per-frame probability -> peaks with NMS
  5  side attribution   which player struck it            (99.4% accurate)
  6  classification     4 classes, temperature-calibrated (T=2.20)
  7  kinematics         13 scalars per shot
  8  rally grouping     gaps > 1.5s                       (validated, F1 0.769)
  9  render             annotated video with live analytics
```

### Performance, from held-out cross-validation

| | |
|---|---|
| Contact detection | **F1 0.881** @ +/-8 frames (67 ms) |
| Side attribution | **0.994** |
| Classification | **macro-F1 0.792** |
| End-to-end | **macro-F1 0.698** |
| Rally boundaries | **F1 0.769** (at the detection ceiling) |

### What it can and cannot label

Per-class thresholds are fitted on held-out data. Two classes are **suppressed** - counted in rallies, never used for coaching, because their precision is too low to trust:

| class | held-out precision | |
|---|---|---|
| serve | 0.974 | coachable, ungated |
| attack | 0.857 when gated | coachable |
| control | caps at 0.804 | **suppressed** |
| defence | caps at 0.556 | **suppressed** |

### Runtime

**~5x realtime on a T4.** `game_2` is 24 minutes, so analysis takes roughly **2 hours** and rendering another **30 minutes**. Pose is cached - re-running is near-instant.

### Requirements

Sideline camera, **120 fps** (the 97-frame window, velocity units and NMS gap are all calibrated to it), both players visible.


## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Install - then **restart the session**

`rtmlib` declares CPU `onnxruntime` as a hard dependency, which shadows `onnxruntime-gpu` and silently drops pose inference to CPU at ~40x the cost. Installing it `--no-deps` with the GPU wheel last is what avoids that.

In [2]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4."
print(f"GPU: {torch.cuda.get_device_name(0)}")

!pip uninstall -y -q onnxruntime onnxruntime-gpu 2>&1 | tail -1
!pip install -q --no-deps rtmlib 2>&1 | tail -1
!pip install -q opencv-python numpy tqdm ultralytics pyarrow 2>&1 | tail -1
!pip install -q "onnxruntime-gpu==1.22.0" 2>&1 | tail -1
!pip list 2>/dev/null | grep -iE "onnxruntime|rtmlib|ultralytics"

import os, glob, site
libs=[]
for sp in site.getsitepackages():
    libs += glob.glob(os.path.join(sp,"nvidia","*","lib"))
if libs: open("/content/_ort_libpath.txt","w").write(":".join(libs))
print("\n"+"="*58)
print("  NOW: Runtime > Restart session, then run from cell 3.")
print("="*58)

GPU: NVIDIA A100-SXM4-40GB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.3/65.3 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.3 MB/s eta 0:00:00
onnxruntime-gpu                       1.22.0
rtmlib                                0.0.16
ultralytics                           8.4.128
ultralytics-platform                  0.1.14
ultralytics-thop                      2.1.6

  NOW: Runtime > Restart session, then run from cell 3.


## 3. Config

In [3]:
BASE       = "/content/drive/MyDrive/tt_coach"
VIDEO_PATH = "/content/drive/MyDrive/tt_coach/raw/videos/game_2.mp4"
VIDEO_ID   = "game_2"

# --- pipeline ---
ACT_STRIDE, ACT_PAD_S, MOTION_THR = 12, 0.75, 0.02
DET_THR, NMS_GAP, RALLY_GAP_S     = 0.60, 30, 1.5     # gap validated F1 0.769
# --- render ---
RALLY_ONLY, STRIDE, OUT_FPS, OUT_SCALE = True, 2, 30, 0.55
MAX_MIN = None          # set e.g. 5 for a quick layout check first

import json, math, shutil, time, os, glob, site, subprocess
from pathlib import Path
import numpy as np, pandas as pd

_lp = Path("/content/_ort_libpath.txt")
if _lp.exists():
    os.environ["LD_LIBRARY_PATH"] = _lp.read_text()+":"+os.environ.get("LD_LIBRARY_PATH","")

import cv2, torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

BASE=Path(BASE); META=BASE/"derived/meta"; CKPT=BASE/"models/checkpoints"
ANALYSED=BASE/"derived/analysed"; ANALYSED.mkdir(parents=True,exist_ok=True)
OUTV=BASE/"outputs/videos"; OUTV.mkdir(parents=True,exist_ok=True)
LOCAL=Path("/content/_work"); LOCAL.mkdir(exist_ok=True)
dev="cuda"

folds=json.loads((META/"folds.json").read_text())
PRE,NF,FPS = folds["window"]["pre"], folds["window"]["n_frames"], 120
CLASSES=["serve","attack","control","defence"]
TECHS=["block","chop","flick","lob","loop","push","serve","smash"]
COL={"serve":(80,220,255),"attack":(80,80,255),
     "control":(120,255,120),"defence":(255,180,60)}      # BGR
L_SHO,R_SHO,L_ELB,R_ELB,L_WRI,R_WRI=5,6,7,8,9,10
L_HIP,R_HIP,L_KNE,R_KNE,L_ANK,R_ANK=11,12,13,14,15,16
FLIP=[(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]
SKEL=[(5,6),(5,7),(7,9),(6,8),(8,10),(5,11),(6,12),(11,12),
      (11,13),(13,15),(12,14),(14,16),(0,5),(0,6)]

CAL=json.loads((META/"calibration.json").read_text())
TEMP=CAL["temperature"]
THRESHOLDS=CAL.get("per_class_thresholds",
                   {"serve":0.25,"attack":0.50,"control":1.01,"defence":1.01})
print(f"temperature {TEMP:.2f}   rally gap {RALLY_GAP_S}s")
print(f"thresholds { {k:round(v,2) for k,v in THRESHOLDS.items()} }")
print(f"suppressed { [k for k,v in THRESHOLDS.items() if v>1.0] }")

temperature 2.20   rally gap 1.5s
thresholds {'serve': 0.25, 'attack': 0.5, 'control': 1.01, 'defence': 1.01}
suppressed ['control', 'defence']


## 4. Load models

In [4]:
class Block(nn.Module):
    def __init__(s,c,d,drop=0.1):
        super().__init__()
        s.c1=nn.Conv1d(c,c,5,padding=2*d,dilation=d); s.c2=nn.Conv1d(c,c,5,padding=2*d,dilation=d)
        s.n1,s.n2=nn.BatchNorm1d(c),nn.BatchNorm1d(c); s.do=nn.Dropout(drop)
    def forward(s,x):
        r=x; x=s.do(F.gelu(s.n1(s.c1(x)))); x=s.do(F.gelu(s.n2(s.c2(x)))); return F.gelu(x+r)

class DetNet(nn.Module):
    def __init__(s,c_in,w=128):
        super().__init__()
        s.stem=nn.Sequential(nn.Conv1d(c_in,w,1),nn.BatchNorm1d(w),nn.GELU())
        s.blocks=nn.Sequential(*[Block(w,d) for d in (1,2,4,8,16,32,64)])
        s.hc,s.hs=nn.Conv1d(w,1,1),nn.Conv1d(w,1,1)
    def forward(s,x):
        z=s.blocks(s.stem(x)); return s.hc(z).squeeze(1), s.hs(z).squeeze(1)

class AttnPool(nn.Module):
    def __init__(s,c):
        super().__init__(); s.score=nn.Conv1d(c,1,1)
    def forward(s,x):
        w=torch.softmax(s.score(x),-1); return torch.cat([(x*w).sum(-1),x.max(-1).values],-1)

class ClsNet(nn.Module):
    def __init__(s,c_in,w=128):
        super().__init__()
        s.stem=nn.Sequential(nn.Conv1d(c_in,w,1),nn.BatchNorm1d(w),nn.GELU())
        s.blocks=nn.Sequential(*[Block(w,d,0.2) for d in (1,2,4,8,16,32)])
        s.pool=AttnPool(w)
        s.trunk=nn.Sequential(nn.Linear(w*2,256),nn.GELU(),nn.Dropout(0.3))
        s.shot,s.tech=nn.Linear(256,4),nn.Linear(256,8)
    def forward(s,x):
        z=s.trunk(s.pool(s.blocks(s.stem(x)))); return s.shot(z), s.tech(z)

dck=torch.load(CKPT/"detector_final.pt",map_location=dev,weights_only=False)
cck=torch.load(CKPT/"classifier_final.pt",map_location=dev,weights_only=False)
DET=DetNet(dck["c_in"]).to(dev); DET.load_state_dict(dck["state"]); DET.eval()
CLS=ClsNet(cck["c_in"]).to(dev); CLS.load_state_dict(cck["state"]); CLS.eval()
DMU,DSD=dck["mu"],dck["sd"]; CMU,CSD=cck["mu"].to(dev),cck["sd"].to(dev)

import onnxruntime as ort
assert "CUDAExecutionProvider" in ort.get_available_providers(), \
    "CUDA provider missing - restart the session after cell 2."
from ultralytics import YOLO
from rtmlib import RTMPose
YDET=YOLO(str(BASE/"models/detector/best.pt")); YDET.to("cuda")
PLAYER_CLS=[k for k,v in YDET.names.items() if v.lower()=="player"][0]
TABLE_CLS =[k for k,v in YDET.names.items() if v.lower()=="table"][0]
POSE=RTMPose(onnx_model=("https://download.openmmlab.com/mmpose/v1/projects/"
     "rtmposev1/onnx_sdk/rtmpose-l_simcc-body7_pt-body7_420e-384x288-"
     "3f5a1437_20230504.zip"),model_input_size=(288,384),
     backend="onnxruntime",device="cuda")
print(f"detector F1 {dck['lovo_f1_at_tol8']} | classifier macro-F1 {cck['lovo_macro_f1']}")

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


Downloading: "https://download.openmmlab.com/mmpose/v1/projects/rtmposev1/onnx_sdk/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.zip" to /root/.cache/rtmlib/hub/checkpoints/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.zip
100%|██████████| 98.9M/98.9M [00:03<00:00, 27.2MB/s]


load /root/.cache/rtmlib/hub/checkpoints/rtmpose-l_simcc-body7_pt-body7_420e-384x288-3f5a1437_20230504.onnx with onnxruntime backend
detector F1 0.881 | classifier macro-F1 0.792


## 5. Pipeline stages

The activity gate decodes **sequentially** - `grab()` advances without converting to BGR, `retrieve()` converts only sampled frames. A random seek per frame forces the decoder back to a keyframe each time; measured 9.2x slower.

In [5]:
def resolve(frames,mid_x,conf=0.35):
    out=[]
    for r in YDET.predict(frames,verbose=False,conf=conf):
        d={"left":None,"right":None}
        if r.boxes is not None and len(r.boxes):
            xy=r.boxes.xyxy.cpu().numpy(); cl=r.boxes.cls.cpu().numpy().astype(int)
            pl=xy[cl==PLAYER_CLS]
            if len(pl):
                cx=(pl[:,0]+pl[:,2])/2; ls,rs=pl[cx<mid_x],pl[cx>=mid_x]
                if len(ls): d["left"]=ls[np.argmin((ls[:,0]+ls[:,2])/2)]
                if len(rs): d["right"]=rs[np.argmax((rs[:,0]+rs[:,2])/2)]
        out.append(d)
    return out

def find_table(cap,n,k=9):
    bx=[]
    for f in np.linspace(n*0.1,n*0.9,k).astype(int):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(f)); ok,fr=cap.read()
        if not ok: continue
        r=YDET.predict(fr,verbose=False,conf=0.35)[0]
        if r.boxes is None or not len(r.boxes): continue
        xy=r.boxes.xyxy.cpu().numpy(); cl=r.boxes.cls.cpu().numpy().astype(int)
        tb=xy[cl==TABLE_CLS]
        if len(tb): bx.append(tb[np.argmax((tb[:,2]-tb[:,0])*(tb[:,3]-tb[:,1]))])
    return np.median(np.stack(bx),0).astype(np.float32) if bx else None

def activity_gate(path,nfr,mid_x):
    cap=cv2.VideoCapture(str(path)); flags,idxs,buf,bidx,prev=[],[],[],[],None
    def flush():
        nonlocal buf,bidx,prev
        for j,d in enumerate(resolve(buf,mid_x)):
            both=d["left"] is not None and d["right"] is not None; mov=True
            if both and prev is not None:
                h=max(d["left"][3]-d["left"][1],1)
                mv=max(abs((d["left"][0]+d["left"][2])/2-prev[0]),
                       abs((d["right"][0]+d["right"][2])/2-prev[1]))/h
                mov=mv>MOTION_THR
            if both: prev=((d["left"][0]+d["left"][2])/2,(d["right"][0]+d["right"][2])/2)
            flags.append(both and mov); idxs.append(bidx[j])
        buf,bidx=[],[]
    pb=tqdm(total=nfr,desc="  gate",leave=False); i=0
    while i<nfr:
        if not cap.grab(): break
        if i%ACT_STRIDE==0:
            ok,fr=cap.retrieve()
            if ok: buf.append(fr); bidx.append(i)
            if len(buf)>=64: flush()
        i+=1
        if i%2000==0: pb.update(2000)
    if buf: flush()
    pb.close(); cap.release()
    pad=int(ACT_PAD_S*FPS); spans=[]
    for k,f in enumerate(flags):
        if not f: continue
        s,e=max(0,idxs[k]-pad),min(nfr-1,idxs[k]+pad)
        if spans and s<=spans[-1][1]+1: spans[-1][1]=max(spans[-1][1],e)
        else: spans.append([s,e])
    return spans

def extract_pose(path,spans):
    total=sum(e-s+1 for s,e in spans)
    F_=np.zeros(total,np.int32); KP=np.zeros((total,2,17,2),np.float16)
    SC=np.zeros((total,2,17),np.float16); BX=np.zeros((total,2,4),np.float16)
    DT=np.zeros((total,2),bool); SG=np.zeros(total,np.int32)
    cap=cv2.VideoCapture(str(path)); w=0
    pb=tqdm(total=total,desc="  pose",leave=False)
    for si,(s0,e0) in enumerate(spans):
        cap.set(cv2.CAP_PROP_POS_FRAMES,int(s0)); pos=s0
        while pos<=e0:
            n=min(600,e0-pos+1); frames=[]
            for _ in range(n):
                ok,fr=cap.read()
                frames.append(fr if ok else (frames[-1] if frames else np.zeros((720,1280,3),np.uint8)))
            n=len(frames)
            di=list(range(0,n,8)); di+=[] if di[-1]==n-1 else [n-1]
            dets=resolve([frames[i] for i in di],mid_x_g)
            boxes={}
            for pi,side in enumerate(["left","right"]):
                kn=[(i,b) for i,b in zip(di,[d[side] for d in dets]) if b is not None]
                if not kn: continue
                ki=np.array([a for a,_ in kn],float); kb=np.stack([b for _,b in kn]).astype(float)
                boxes[pi]=np.stack([np.interp(np.arange(n),ki,kb[:,c]) for c in range(4)],1).astype(np.float32)
            for k in range(n):
                bb,who=[],[]
                for pi in (0,1):
                    if pi in boxes:
                        b=boxes[pi][k]; bw,bh=b[2]-b[0],b[3]-b[1]
                        b=np.array([max(0,b[0]-bw*.18),max(0,b[1]-bh*.11),b[2]+bw*.18,b[3]+bh*.045],np.float32)
                        bb.append(b); who.append(pi); BX[w+k,pi]=b; DT[w+k,pi]=True
                if bb:
                    kp,sc=POSE(frames[k],bboxes=np.stack(bb))
                    for j,pi in enumerate(who): KP[w+k,pi]=kp[j]; SC[w+k,pi]=sc[j]
                F_[w+k]=pos+k; SG[w+k]=si
            w+=n; pos+=n; pb.update(n); del frames
    pb.close(); cap.release()
    return dict(frame_idx=F_[:w],seg_id=SG[:w],keypoints=KP[:w],
                scores=SC[:w],boxes=BX[:w],detected=DT[:w])

def torso_scale(kp):
    t=np.linalg.norm((kp[:,L_SHO]+kp[:,R_SHO])/2-(kp[:,L_HIP]+kp[:,R_HIP])/2,axis=-1)
    t=t[t>1]; return float(np.median(t)) if t.size else 1.0

def canon(kp,sc,seg,mirror):
    kp=kp.astype(np.float32).copy()
    hip=(kp[:,L_HIP]+kp[:,R_HIP])/2; sho=(kp[:,L_SHO]+kp[:,R_SHO])/2
    torso=np.linalg.norm(sho-hip,axis=-1); scale=np.ones(len(kp),np.float32)
    for s in np.unique(seg):
        m=seg==s; t=torso[m]; t=t[t>1]; scale[m]=np.median(t) if t.size else 1.
    kp=(kp-hip[:,None,:])/np.maximum(scale,1e-3)[:,None,None]
    if mirror:
        kp[...,0]*=-1; sc=sc.copy()
        for a,b in FLIP: kp[:,[a,b]]=kp[:,[b,a]]; sc[:,[a,b]]=sc[:,[b,a]]
    return kp,sc,scale

def build_stream(raw,table):
    seg=raw["seg_id"]; KP=raw["keypoints"]; SC=raw["scores"]; DT=raw["detected"]
    ch,kps,vals,tds=[],[],[],[]
    for pi in (0,1):
        kp,sc,scale=canon(KP[:,pi],SC[:,pi].astype(np.float32),seg,mirror=(pi==1))
        vel=np.zeros_like(kp); vel[1:]=np.diff(kp,axis=0)
        vel[np.diff(seg,prepend=seg[0])!=0]=0
        ch+=[kp.reshape(len(kp),-1),vel.reshape(len(kp),-1),(sc*DT[:,pi:pi+1]).astype(np.float32)]
        kps.append(kp); vals.append((sc>=.35)&DT[:,pi:pi+1])
        hx=(KP[:,pi,L_HIP,0]+KP[:,pi,R_HIP,0])/2
        edge=table[0] if pi==0 else table[2]
        tds.append(np.abs(hx-edge)/np.maximum(scale,1e-3)
                   if table is not None and table[2]>table[0] else np.zeros(len(kp),np.float32))
    cuts=np.where(np.diff(seg)!=0)[0]+1; b=np.concatenate([[0],cuts,[len(seg)]])
    return dict(X=np.nan_to_num(np.concatenate(ch,1).astype(np.float32)),
                kp=np.stack(kps,1),val=np.stack(vals,1),td=np.nan_to_num(np.stack(tds,1)),
                fidx=raw["frame_idx"],scores=SC,detected=DT,
                spans=[(int(b[i]),int(b[i+1])) for i in range(len(b)-1)])

def decode(prob,thr=DET_THR,gap=NMS_GAP):
    idx=np.where(prob>=thr)[0]
    if not len(idx): return np.array([],int)
    pk=[i for i in idx if prob[i]==prob[max(0,i-gap//2):i+gap//2+1].max()]
    pk=sorted(pk,key=lambda i:-prob[i]); keep=[]
    for p in pk:
        if all(abs(p-k)>=gap for k in keep): keep.append(p)
    return np.array(sorted(keep),int)

def window_at(st,idx,side):
    sel=np.clip(np.arange(idx-PRE,idx-PRE+NF),0,len(st["X"])-1)
    pi=0 if side=="left" else 1
    kp=st["kp"][sel,pi]; val=st["val"][sel,pi].astype(np.float32)
    vel=np.zeros_like(kp); vel[1:]=np.diff(kp,axis=0)
    x=np.concatenate([kp.reshape(NF,-1),vel.reshape(NF,-1),val,st["td"][sel,pi][:,None]],1)
    return np.nan_to_num(x).T.astype(np.float32),kp,val,sel

def angle(a,b,c):
    v1,v2=a-b,c-b
    cs=(v1*v2).sum(-1)/np.maximum(np.linalg.norm(v1,axis=-1)*np.linalg.norm(v2,axis=-1),1e-6)
    return np.degrees(np.arccos(np.clip(cs,-1,1)))

def kinematics(kp,val,td_win,wri):
    h=-kp[...,1]; vel=np.zeros_like(kp); vel[1:]=np.diff(kp,axis=0)
    spd=np.linalg.norm(vel,axis=-1); spd[~val.astype(bool)]=np.nan
    w=spd[:,wri]; pre,post=slice(0,PRE),slice(PRE,NF)
    sh,el=(R_SHO,R_ELB) if wri==R_WRI else (L_SHO,L_ELB)
    d_hip=np.linalg.norm(kp[:,wri],axis=-1); ea=angle(kp[:,sh],kp[:,el],kp[:,wri])
    pk=np.nanmax(w) if np.isfinite(w).any() else np.nan; rec=np.nan
    if np.isfinite(pk) and pk>0:
        a=np.where(np.nan_to_num(w[post])<0.2*pk)[0]; rec=float(a[0]) if len(a) else np.nan
    tr=(kp[:,L_SHO]+kp[:,R_SHO])/2; ta=np.degrees(np.arctan2(tr[:,0],-tr[:,1]))
    return {"backswing_amplitude":float(np.nanmax(d_hip[pre])),"peak_wrist_speed":float(pk),
        "time_to_peak":int(np.nanargmax(w)-PRE) if np.isfinite(w).any() else None,
        "contact_height":float(h[PRE,wri]-(h[PRE,L_SHO]+h[PRE,R_SHO])/2),
        "elbow_angle":float(ea[PRE]),"elbow_range":float(np.nanmax(ea)-np.nanmin(ea)),
        "trunk_lean":float(ta[PRE]),"trunk_rotation":float(np.nanmax(ta)-np.nanmin(ta)),
        "table_distance":float(td_win[PRE]),
        "stance_width":float(abs(kp[PRE,L_ANK,0]-kp[PRE,R_ANK,0])),
        "knee_angle":float((angle(kp[PRE:PRE+1,L_HIP],kp[PRE:PRE+1,L_KNE],kp[PRE:PRE+1,L_ANK])[0]+
                            angle(kp[PRE:PRE+1,R_HIP],kp[PRE:PRE+1,R_KNE],kp[PRE:PRE+1,R_ANK])[0])/2),
        "follow_through":float(np.nansum(w[post])),"recovery_time":rec}
print("stages defined")

stages defined


## 6. `analyse()`

In [6]:
def analyse(video_path,video_id=None,force=False):
    global mid_x_g
    video_path=Path(video_path); vid=video_id or video_path.stem
    outdir=ANALYSED/vid; outdir.mkdir(parents=True,exist_ok=True)
    cache=outdir/"pose_raw.npz"; t0=time.time()

    local=LOCAL/video_path.name
    if not local.exists():
        print("  copying video to local disk ..."); shutil.copy(video_path,local)
    cap=cv2.VideoCapture(str(local))
    nfr=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)); fps=cap.get(cv2.CAP_PROP_FPS)
    W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); table=find_table(cap,nfr); cap.release()
    mid_x_g=(table[0]+table[2])/2 if table is not None else W/2
    dur=nfr/max(fps,1)
    print(f"{vid}: {nfr:,} frames @ {fps:.0f}fps = {dur/60:.1f} min")
    if abs(fps-120)>5:
        print(f"  !! {fps:.0f}fps - the pipeline is calibrated for 120fps. "
              f"Windows, velocities and NMS will not mean what they should.")

    if cache.exists() and not force:
        raw={k:v for k,v in np.load(cache,allow_pickle=True).items()}
        table=raw.pop("table_box",table)
        print(f"  cached pose: {len(raw['frame_idx']):,} frames")
    else:
        ta=time.time(); spans=activity_gate(local,nfr,mid_x_g)
        cov=sum(e-s+1 for s,e in spans)/max(nfr,1)
        print(f"  [1] gate  {time.time()-ta:.0f}s -> {len(spans)} regions, {cov:.0%} active")
        ta=time.time(); raw=extract_pose(local,spans)
        print(f"  [2] pose  {time.time()-ta:.0f}s -> {len(raw['frame_idx']):,} frames")
        np.savez_compressed(cache,**raw,
            table_box=table if table is not None else np.zeros(4,np.float32))
    st=build_stream(raw,table)
    print(f"  [3] canonicalised")

    prob=np.zeros(len(st["X"]),np.float32); sidep=np.zeros(len(st["X"]),np.float32)
    with torch.no_grad():
        for a,b in st["spans"]:
            x=torch.tensor(((st["X"][a:b]-DMU)/DSD).T[None],dtype=torch.float32,device=dev)
            lc,ls=DET(x)
            prob[a:b]=torch.sigmoid(lc)[0].cpu().numpy()
            sidep[a:b]=torch.sigmoid(ls)[0].cpu().numpy()
    peaks=decode(prob)
    print(f"  [4] contacts: {len(peaks)}")
    if not len(peaks): return pd.DataFrame(),None
    sides=["right" if sidep[i]>=0.5 else "left" for i in peaks]
    print(f"  [5] sides: {sides.count('left')} left / {sides.count('right')} right")

    wins,kps,vals,sels=[],[],[],[]
    for i,s in zip(peaks,sides):
        x,kp,val,sel=window_at(st,i,s); wins.append(x);kps.append(kp);vals.append(val);sels.append(sel)
    with torch.no_grad():
        lo,lt=CLS((torch.tensor(np.stack(wins),device=dev)-CMU)/CSD)
        pr=torch.softmax(lo/TEMP,1).cpu().numpy(); pt=torch.softmax(lt/TEMP,1).cpu().numpy()
    print(f"  [6] classified (T={TEMP:.2f})")

    frames=st["fidx"][peaks]; rid,sidx,cur,last=[],[],0,None
    for f in frames:
        if last is not None and (f-last)/FPS>RALLY_GAP_S: cur+=1; k=0
        else: k=0 if last is None else sidx[-1]+1
        rid.append(cur); sidx.append(k); last=f
    rlen=pd.Series(rid).value_counts().to_dict()

    rows=[]
    for n,(i,s) in enumerate(zip(peaks,sides)):
        pi=0 if s=="left" else 1; cl=CLASSES[int(pr[n].argmax())]
        km=kinematics(kps[n],vals[n],st["td"][sels[n],pi],R_WRI if s=="left" else L_WRI)
        rows.append(dict(video_id=vid,rally_id=rid[n],shot_index=sidx[n],player=s,
            frame=int(frames[n]),timestamp_s=float(frames[n]/FPS),shot_class=cl,
            class_confidence=float(pr[n].max()),technique=TECHS[int(pt[n].argmax())],
            abstain=bool(pr[n].max()<THRESHOLDS[cl]),detect_confidence=float(prob[i]),
            rally_length=int(rlen[rid[n]]),
            pose_confidence=float(st["scores"][sels[n],pi][st["scores"][sels[n],pi]>0].mean()
                                  if (st["scores"][sels[n],pi]>0).any() else 0.),
            detected=float(st["detected"][sels[n],pi].mean()),**km))
    df=pd.DataFrame(rows)
    print(f"  [7] kinematics + [8] {df.rally_id.nunique()} rallies")

    df.to_parquet(outdir/"shots.parquet",index=False)
    np.savez_compressed(outdir/"pose_windows.npz",
        stroke_id=df.apply(lambda r:f"{vid}_{r.frame:07d}",axis=1).values.astype(str),
        pose_window=np.stack(kps).astype(np.float16),valid=np.stack(vals))
    el=time.time()-t0
    print(f"\n  {len(df)} shots, {df.rally_id.nunique()} rallies, "
          f"{df.abstain.sum()} suppressed  [{el/60:.0f} min = {el/max(dur,1):.1f}x realtime]")
    return df,el/max(dur,1)
print("analyse() ready")

analyse() ready


## 7. Run

`game_2` is 24 minutes - expect **~2 hours**. Pose is cached, so a second run skips straight to detection.

In [ ]:
shots,rt = analyse(VIDEO_PATH, video_id=VIDEO_ID, force=False)
shots.head(10)[["rally_id","shot_index","player","timestamp_s","shot_class",
                "class_confidence","abstain","peak_wrist_speed"]]

  copying video to local disk ...
game_2: 172,200 frames @ 120fps = 23.9 min


  gate:   0%|          | 0/172200 [00:00<?, ?it/s]

  [1] gate  399s -> 138 regions, 66% active


  pose:   0%|          | 0/114258 [00:00<?, ?it/s]

  [2] pose  2052s -> 114,258 frames
  [3] canonicalised
  [4] contacts: 369
  [5] sides: 186 left / 183 right
  [6] classified (T=2.20)
  [7] kinematics + [8] 102 rallies

  369 shots, 102 rallies, 115 suppressed  [45 min = 1.9x realtime]


,rally_id,shot_index,player,timestamp_s,shot_class,class_confidence,abstain,peak_wrist_speed
0,0,0,right,7.291667,serve,0.995014,False,0.146673
1,0,1,left,8.041667,control,0.940582,True,0.073848
2,0,2,right,8.675000,control,0.693967,True,0.045910
3,0,3,left,9.250000,attack,0.899824,False,0.149713
4,1,0,right,17.483333,serve,0.995687,False,0.491234
5,1,1,left,17.991667,attack,0.505618,False,0.070761
6,1,2,right,18.458333,attack,0.936182,False,0.125175
7,2,0,left,28.733333,serve,0.989669,False,0.701591
8,2,1,right,29.333333,attack,0.692765,False,0.105888
9,3,0,left,40.583333,serve,0.989186,False,0.097989


## 8. Match summary

In [ ]:
N_RALLY=shots.rally_id.nunique()
rl=shots.groupby("rally_id").size()
print("="*62); print(f"  {VIDEO_ID}"); print("="*62)
print(f"  shots    {len(shots)}")
print(f"  rallies  {N_RALLY}   mean {rl.mean():.1f}, longest {rl.max()}")
print(f"  coachable {(~shots.abstain).sum()} ({(~shots.abstain).mean():.0%})")
print(f"\n  {'':<10}{'LEFT':>10}{'RIGHT':>10}")
for c in CLASSES:
    l=((shots.player=="left")&(shots.shot_class==c)).sum()
    r=((shots.player=="right")&(shots.shot_class==c)).sum()
    tag="" if THRESHOLDS[c]<=1.0 else "  (suppressed)"
    print(f"  {c:<10}{l:>10}{r:>10}{tag}")
print(f"  {'TOTAL':<10}{(shots.player=='left').sum():>10}"
      f"{(shots.player=='right').sum():>10}")
for side in ("left","right"):
    s=shots[shots.player==side]
    if len(s):
        print(f"\n  {side.upper()}: peak wrist speed {s.peak_wrist_speed.mean():.3f}, "
              f"backswing {s.backswing_amplitude.mean():.2f}, "
              f"table dist {s.table_distance.mean():.2f}")
print("="*62)

  game_2
  shots    369
  rallies  102   mean 3.6, longest 10
  coachable 254 (69%)

                  LEFT     RIGHT
  serve             43        48
  attack            98        68
  control           27        39  (suppressed)
  defence           18        28  (suppressed)
  TOTAL            186       183

  LEFT: peak wrist speed 0.265, backswing 1.26, table dist 1.15

  RIGHT: peak wrist speed 0.292, backswing 1.35, table dist 1.15


## 9. Render

Live analytics accumulate as the match plays: **left player bottom-left, right player bottom-right, rally counter top-centre.**

Dead time is skipped - rendering all 172,200 frames would take hours for footage that is mostly players standing around. Output is half speed, because a table-tennis stroke at 120 fps real time is a blur.

Set `MAX_MIN = 5` in cell 3 for a quick layout check before the full run.

In [ ]:
raw=np.load(ANALYSED/VIDEO_ID/"pose_raw.npz",allow_pickle=True)
FIDX,KPr,SCr,BXr,DTr=(raw["frame_idx"],raw["keypoints"].astype(np.float32),
    raw["scores"].astype(np.float32),raw["boxes"].astype(np.float32),raw["detected"])
TABLE=raw["table_box"] if "table_box" in raw else np.zeros(4,np.float32)
F2I={int(f):i for i,f in enumerate(FIDX)}

WSPD=np.zeros((len(FIDX),2),np.float32)
for pi in (0,1):
    ts=torso_scale(KPr[:,pi])
    for w in (L_WRI,R_WRI):
        v=np.zeros(len(FIDX),np.float32)
        v[1:]=np.linalg.norm(np.diff(KPr[:,pi,w],axis=0),axis=-1)
        v[SCr[:,pi,w]<0.35]=0
        WSPD[:,pi]=np.maximum(WSPD[:,pi],v/max(ts,1e-3))

def box(img,x0,y0,x1,y1,a=.74):
    ov=img.copy(); cv2.rectangle(ov,(x0,y0),(x1,y1),(22,22,22),-1)
    cv2.addWeighted(ov,a,img,1-a,0,img)
def put(img,t,x,y,col=(255,255,255),sc=.55,th=1):
    cv2.putText(img,t,(x,y),cv2.FONT_HERSHEY_SIMPLEX,sc,(0,0,0),th+3)
    cv2.putText(img,t,(x,y),cv2.FONT_HERSHEY_SIMPLEX,sc,col,th)

def player_panel(img,side,st,corner):
    h,w=img.shape[:2]; pw,ph=430,250
    x0=20 if corner=="bl" else w-pw-20; y0=h-ph-20
    box(img,x0,y0,x0+pw,y0+ph)
    col=(120,255,120) if side=="left" else (60,180,255)
    cv2.rectangle(img,(x0,y0),(x0+pw,y0+ph),col,2)
    put(img,f"{side.upper()} PLAYER",x0+16,y0+32,col,.72,2)
    put(img,f"{st['n']} shots",x0+pw-135,y0+32,(220,220,220),.6,1)
    y=y0+64
    for c in CLASSES:
        n=st["cls"][c]; pct=n/max(st["n"],1)
        put(img,f"{c:<8}",x0+16,y+4,COL[c],.55,1)
        put(img,f"{n:>3}",x0+130,y+4,COL[c],.55,2)
        bx0,bw=x0+180,170
        cv2.rectangle(img,(bx0,y-9),(bx0+bw,y+3),(55,55,55),-1)
        cv2.rectangle(img,(bx0,y-9),(bx0+int(bw*pct),y+3),COL[c],-1)
        put(img,f"{pct*100:>3.0f}%",bx0+bw+12,y+4,(200,200,200),.48,1)
        y+=26
    cv2.line(img,(x0+16,y-2),(x0+pw-16,y-2),(70,70,70),1)
    put(img,f"peak wrist speed  {st['speed']:.3f}",x0+16,y+20,(210,210,210),.5,1)
    put(img,f"backswing         {st['back']:.2f}", x0+16,y+42,(210,210,210),.5,1)
    put(img,f"coachable         {st['coach']:.0%}",x0+16,y+64,(150,230,150),.5,1)
    return img

def rally_banner(img,r,shot,rlen):
    w=img.shape[1]; bw,bh=420,96; x0=(w-bw)//2
    box(img,x0,16,x0+bw,16+bh,.78)
    cv2.rectangle(img,(x0,16),(x0+bw,16+bh),(0,220,220),2)
    put(img,f"RALLY  {r+1} / {N_RALLY}",x0+26,62,(0,255,255),1.0,2)
    put(img,f"shot {shot+1} of {rlen}",x0+26,92,(200,200,200),.58,1)
    return img

def trace(img,frame):
    w=img.shape[1]; pw,ph=340,92; x0,y0=w-pw-20,20
    box(img,x0,y0,x0+pw,y0+ph)
    lo,hi=frame-int(2.5*FPS),frame
    idx=[F2I.get(f) for f in range(lo,hi,2)]
    vals=np.array([[WSPD[i,0],WSPD[i,1]] if i is not None else [0.,0.] for i in idx],np.float32)
    if vals.size:
        m=float(vals.max()) or 1.0
        for pi,c in ((0,(120,255,120)),(1,(60,180,255))):
            pts=[(int(x0+pw*k/max(len(vals),1)),int(y0+ph-ph*.85*vals[k,pi]/m))
                 for k in range(len(vals))]
            if len(pts)>1: cv2.polylines(img,[np.array(pts,np.int32)],False,c,2)
    for _,s in shots[(shots.frame>=lo)&(shots.frame<hi)].iterrows():
        x=int(x0+pw*(s.frame-lo)/max(hi-lo,1))
        cv2.line(img,(x,y0+8),(x,y0+ph),COL[s.shot_class],2)
    put(img,"wrist speed",x0+8,y0+16,(170,170,170),.42,1)
    return img

def blank(): return dict(n=0,cls={c:0 for c in CLASSES},speed=0.,back=0.,coach=0.,
                         _sp=[],_bk=[],_ab=[])
def upd(st,s):
    st["n"]+=1; st["cls"][s.shot_class]+=1
    if np.isfinite(s.peak_wrist_speed): st["_sp"].append(s.peak_wrist_speed)
    if np.isfinite(s.backswing_amplitude): st["_bk"].append(s.backswing_amplitude)
    st["_ab"].append(0 if s.abstain else 1)
    st["speed"]=float(np.mean(st["_sp"])) if st["_sp"] else 0.
    st["back"] =float(np.mean(st["_bk"])) if st["_bk"] else 0.
    st["coach"]=float(np.mean(st["_ab"])) if st["_ab"] else 0.

if RALLY_ONLY:
    SEGS=[]
    for r,g in shots.groupby("rally_id"):
        s=int(g.frame.min()-1.0*FPS); e=int(g.frame.max()+1.0*FPS)
        if SEGS and s<=SEGS[-1][1]: SEGS[-1]=(SEGS[-1][0],max(SEGS[-1][1],e))
        else: SEGS.append((s,e))
else:
    SEGS=[(int(shots.frame.min()),int(shots.frame.max()))]
total=sum((e-s)//STRIDE for s,e in SEGS)
if MAX_MIN:
    cap_f=int(MAX_MIN*60*OUT_FPS); keep,acc=[],0
    for s,e in SEGS:
        n=(e-s)//STRIDE
        if acc+n>cap_f: break
        keep.append((s,e)); acc+=n
    SEGS,total=keep,acc
print(f"{len(SEGS)} segments, {total:,} frames -> {total/OUT_FPS/60:.1f} min video")

src=LOCAL/Path(VIDEO_PATH).name
cap=cv2.VideoCapture(str(src))
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)*OUT_SCALE)
H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)*OUT_SCALE)
tmp=Path("/content/_full.mp4")
vw=cv2.VideoWriter(str(tmp),cv2.VideoWriter_fourcc(*"mp4v"),OUT_FPS,(W,H))
ST={"left":blank(),"right":blank()}
rlen=shots.groupby("rally_id").size().to_dict()
flash,rnow,snow={},0,0
pb=tqdm(total=total,desc="render")
for s0,e0 in SEGS:
    cap.set(cv2.CAP_PROP_POS_FRAMES,max(0,s0)); f=max(0,s0)
    while f<e0:
        ok,img=cap.read()
        if not ok: break
        if (f-max(0,s0))%STRIDE: f+=1; continue
        if TABLE[2]>TABLE[0]:
            cv2.rectangle(img,(int(TABLE[0]),int(TABLE[1])),
                          (int(TABLE[2]),int(TABLE[3])),(255,255,0),2)
        i=F2I.get(f)
        if i is not None:
            for pi,col in ((0,(120,255,120)),(1,(60,180,255))):
                if not DTr[i,pi]: continue
                b=BXr[i,pi].astype(int)
                cv2.rectangle(img,(b[0],b[1]),(b[2],b[3]),col,2)
                kp,sc=KPr[i,pi],SCr[i,pi]
                for a,bb in SKEL:
                    if sc[a]>.3 and sc[bb]>.3:
                        cv2.line(img,tuple(kp[a].astype(int)),tuple(kp[bb].astype(int)),col,2)
                for j in range(17):
                    if sc[j]>.3:
                        cv2.circle(img,tuple(kp[j].astype(int)),4,
                                   (0,0,255) if j in (L_WRI,R_WRI) else col,-1)
        for _,s in shots[(shots.frame>=f)&(shots.frame<f+STRIDE)].iterrows():
            upd(ST[s.player],s); rnow,snow=int(s.rally_id),int(s.shot_index)
            flash[int(s.frame)]=s
        for ff in list(flash):
            s=flash[ff]; age=f-ff
            if age>FPS*0.5: flash.pop(ff); continue
            i2=F2I.get(int(s.frame))
            if i2 is None: continue
            pi=0 if s.player=="left" else 1
            b=BXr[i2,pi].astype(int); cx,cy=(b[0]+b[2])//2,max(b[1]-30,130)
            cv2.circle(img,(cx,cy),int(26+age*.7),COL[s.shot_class],3)
            put(img,s.shot_class.upper()+("" if not s.abstain else " (?)"),
                cx-70,cy-40,COL[s.shot_class],.78,2)
        img=trace(img,f)
        img=rally_banner(img,rnow,snow,rlen.get(rnow,0))
        img=player_panel(img,"left",ST["left"],"bl")
        img=player_panel(img,"right",ST["right"],"br")
        put(img,f"{int(f/FPS//60):02d}:{(f/FPS)%60:04.1f}",24,44,(200,200,200),.6,1)
        vw.write(cv2.resize(img,(W,H))); pb.update(1); f+=1
pb.close(); vw.release(); cap.release()
out=OUTV/f"{VIDEO_ID}_analysed.mp4"
subprocess.run(["ffmpeg","-y","-loglevel","error","-i",str(tmp),"-c:v","libx264",
                "-preset","fast","-crf","26","-pix_fmt","yuv420p",str(out)],check=True)
tmp.unlink(missing_ok=True)
print(f"\n-> {out}  ({out.stat().st_size/1e6:.0f} MB, {total/OUT_FPS/60:.1f} min)")

### Play

In [ ]:
from IPython.display import HTML
from base64 import b64encode
mb=out.stat().st_size/1e6
if mb>90:
    print(f"{mb:.0f} MB - too large to embed. Open from Drive:\n  {out}")
else:
    HTML(f"""<video width=920 controls loop>
<source src="data:video/mp4;base64,{b64encode(out.read_bytes()).decode()}" type="video/mp4"></video>
<p style="font-family:monospace;font-size:12px">
cyan = table | green = left player | blue = right player | red dots = wrists |
(?) = class suppressed for coaching</p>""")

---
## Outputs

| file | |
|---|---|
| `derived/analysed/{id}/shots.parquet` | one row per shot: class, confidence, player, rally, 13 kinematics |
| `derived/analysed/{id}/pose_windows.npz` | 97x17x2 canonical window per shot - Phase 3's raw material |
| `derived/analysed/{id}/pose_raw.npz` | cached pose, so re-running skips extraction |
| `outputs/videos/{id}_analysed.mp4` | annotated video |

**To analyse a different video:** change `VIDEO_PATH` and `VIDEO_ID` in cell 3 and re-run from cell 7.

**Known constraints:** sideline camera only, 120 fps assumed, `control` and `defence` labels are counted but not coachable.
